# 🏭 Project 24 · pH Control Intelligence — Grand Finale
## Model-Based Reinforcement Learning for Chemical Reactor Control

**Reinforcement Learning Family | Operational Excellence Portfolio | LozanoLsa**

---

*Twenty-three projects.*

From the first logistic regression — trained on labeled examples,
learning to classify defects from historical data —
to this: an agent that builds a **mental model of the world** from data,
simulates possible futures through that model,
and chooses the action predicted to produce the best outcome.

Not memorizing values in a table.
Not following the gradient of past rewards.
**Planning.**

---

## The Architecture of Model-Based RL

Three components work together:

$$\underbrace{s_t, a_t \xrightarrow{\hat{f}_\theta} \hat{s}_{t+1}}_{\text{World Model}} \quad \xrightarrow{\text{simulate H steps}} \quad \underbrace{\hat{G}_t = \sum_{h=0}^{H} \gamma^h \hat{r}_{t+h}}_{\text{Planner}} \quad \xrightarrow{\arg\max_a} \quad \underbrace{a_t^*}_{\text{Controller}}$$

1. **World Model** $\hat{f}_\theta$: learns $\hat{pH}_{t+1} = f(\text{pH}_t, T_t, V_t, \text{buffer}, \text{dose})$ from offline data
2. **Planner**: for each action, simulates $H$ steps ahead, computes expected return $\hat{G}_t$
3. **Controller**: executes $a^* = \arg\max_a \hat{G}_t$ in the real environment

The key advantage over reactive methods: the agent does not wait to observe the consequences of a decision.
It **simulates** them first — running the world model forward, evaluating all futures,
choosing the path closest to pH = 7.0.

---

## The Final Industrial Problem: pH Control in a Chemical Reactor

| Variable | Unit | Role |
|----------|------|------|
| `ph_t` | pH | Current acidity/alkalinity of the reactor |
| `temp_t_c` | °C | Reaction temperature (20–30°C) |
| `volume_t_l` | L | Reactor volume (900–1100 L) |
| `buffer_capacity` | — | Buffer strength (0.8–1.5, resists pH change) |

| Action | Dose | Effect |
|--------|------|--------|
| 0 — No dose | 0 ml | Baseline drift only |
| 1 — Small base | +0.5 ml | Gently raises pH |
| 2 — Large base | +1.0 ml | Strongly raises pH |
| 3 — Small acid | −0.5 ml | Gently lowers pH |
| 4 — Large acid | −1.0 ml | Strongly lowers pH |

**Target**: pH = 7.0 ± 0.2 (acceptable window: [6.8, 7.2])
**Penalty**: Deviation from target → product off-spec, potential equipment damage

A random dosing policy hits the target window **3.5% of steps**.
The model-based agent reaches **10.6%** — a **3× improvement** through planning.

---
**LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa**


## 2 · Setup

In [ ]:
# ── 2 · Setup ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d", "axes.labelcolor": "#c9d1d9",
    "xtick.color": "#8b949e", "ytick.color": "#8b949e",
    "text.color": "#c9d1d9", "grid.color": "#21262d",
    "grid.linestyle": "--", "grid.alpha": 0.4,
    "legend.facecolor": "#161b22", "legend.edgecolor": "#30363d",
})

C_BLUE = "#4C9BE8"; C_RED = "#E8574C"; C_OK = "#3FB950"
C_WARN = "#F5A623"; C_BG = "#0d1117"; C_CARD = "#161b22"
C_MUTED = "#8b949e"

# ── Process constants ─────────────────────────────────────────────────────────
PH_TARGET = 7.0
PH_LOW    = 6.8
PH_HIGH   = 7.2
ACTIONS   = {0: 0.0, 1: +0.5, 2: +1.0, 3: -0.5, 4: -1.0}
ACTION_NAMES = ["No dose","Base +0.5ml","Base +1.0ml","Acid -0.5ml","Acid -1.0ml"]

print("Model-Based RL pipeline initialized.")
print(f"  Target pH : {PH_TARGET} ± 0.2  [{PH_LOW}, {PH_HIGH}]")
print(f"  Actions   : {len(ACTIONS)} (no dose / base / acid in 2 magnitudes)")

## 3 · Load Data — Offline Transition Dataset

The world model is trained **offline** from a historical dataset of transitions:
$(s_t, a_t, r_t, s_{t+1})$ collected from 2,000 episodes under a random dosing policy.

This is the **offline RL** paradigm: we learn a model of the world without any additional
interaction with the real reactor. The world model is then used to plan future actions
without exposing the reactor to suboptimal random exploration.

| Column | Description |
|--------|-------------|
| `ph_t` / `ph_t1` | pH before / after action |
| `temp_t_c` / `temp_t1_c` | Temperature (°C) before / after |
| `volume_t_l` / `volume_t1_l` | Volume (L) before / after |
| `buffer_capacity` | Buffer strength — how much the pH resists change |
| `action` | Action index (0–4) |
| `dose_ml` | Actual dose applied (ml) |
| `reward` | Reward received after action |
| `done` | Whether episode ended at this step |


In [ ]:
try:
    df = pd.read_csv("Data_pH.csv")
except FileNotFoundError:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/LozanoLsa/"
        "ModelBased_pH/main/Data_pH.csv"
    )

features_wm = ["ph_t", "temp_t_c", "volume_t_l", "buffer_capacity", "dose_ml"]
target_wm   = "ph_t1"

print(f"Dataset: {df.shape[0]:,} transitions | {df['episode'].nunique():,} episodes")
print(f"Steps per episode: up to {df['step'].max()+1}")
print(f"\nBaseline (random policy):")
print(f"  Mean reward/step : {df['reward'].mean():.3f}")
print(f"  Steps in target  : {((df['ph_t1']>=PH_LOW)&(df['ph_t1']<=PH_HIGH)).mean():.1%}")
print(f"  pH range observed: [{df['ph_t'].min():.2f}, {df['ph_t'].max():.2f}]")
df.head(8)

## 4 · Exploratory Data Analysis

Understanding the process dynamics before fitting any model:

1. **pH starting distribution** — where does the reactor begin each episode?
2. **Reward landscape** — how does pH deviation map to reward?
3. **pH response to each action** — what is the causal effect of each dosing decision?


In [ ]:
# ── EDA 1: pH distribution + reward landscape ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor(C_BG)
fig.suptitle("Offline Dataset — pH Reactor Dynamics", color="white", fontsize=13, y=1.01)

# pH_t distribution
ax = axes[0]
ax.hist(df["ph_t"], bins=50, color=C_BLUE, edgecolor=C_BG, alpha=0.85)
ax.axvline(PH_TARGET, color=C_OK, lw=2, ls="--", label=f"Target pH={PH_TARGET}")
ax.axvspan(PH_LOW, PH_HIGH, alpha=0.15, color=C_OK, label="Acceptable [6.8–7.2]")
ax.set_xlabel("pH"); ax.set_ylabel("Count")
ax.set_title("pH Starting Distribution\n(random episodes)", color="white"); ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Reward vs pH
ax = axes[1]
ph_range = np.linspace(3, 9, 200)
rewards_curve = [-10*abs(p-PH_TARGET) + (5 if PH_LOW<=p<=PH_HIGH else 0) for p in ph_range]
ax.plot(ph_range, rewards_curve, color=C_BLUE, lw=2)
ax.axvspan(PH_LOW, PH_HIGH, alpha=0.2, color=C_OK)
ax.axvline(PH_TARGET, color=C_OK, lw=1.5, ls="--")
ax.set_xlabel("pH"); ax.set_ylabel("Reward (no dosing cost)")
ax.set_title("Reward Function vs pH\n(zero dose)", color="white")
ax.grid(True, alpha=0.3)

# pH response per action
ax = axes[2]
delta_by_action = df.groupby("action").apply(lambda g: (g["ph_t1"]-g["ph_t"]).mean())
colors_a = [C_MUTED if d==0 else (C_OK if d>0 else C_RED) for d in delta_by_action.values
            ] if False else [C_BLUE]*5
ax.bar(range(5), delta_by_action.values,
       color=[C_OK if d>0.01 else (C_RED if d<-0.01 else C_MUTED) for d in delta_by_action.values],
       edgecolor=C_BG, alpha=0.85)
ax.set_xticks(range(5)); ax.set_xticklabels([n.replace(" ","\n") for n in ACTION_NAMES], fontsize=7)
ax.axhline(0, color="#8b949e", lw=1)
ax.set_ylabel("Mean ΔpH"); ax.set_title("Mean pH Change\nper Action (causal)", color="white")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_01_eda.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Green bars = raise pH (base addition). Red bars = lower pH (acid addition).")
print("Asymmetry reflects the non-linear pH dynamics and buffer resistance.")

In [ ]:
# ── EDA 2: Sample pH trajectories under random policy ────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor(C_BG)

sample_eps = df["episode"].unique()[:12]
for ep in sample_eps:
    ep_data = df[df["episode"]==ep].sort_values("step")
    ax.plot(ep_data["step"], ep_data["ph_t"], lw=1, alpha=0.5, color=C_BLUE)

ax.axhspan(PH_LOW, PH_HIGH, alpha=0.15, color=C_OK, label="Target zone [6.8–7.2]")
ax.axhline(PH_TARGET, color=C_OK, lw=2, ls="--", label=f"Target pH={PH_TARGET}")
ax.set_xlabel("Step"); ax.set_ylabel("pH")
ax.set_title("pH Trajectories — 12 Sample Episodes (Random Policy)\n"
             "Most episodes never reach the target zone",
             color="white", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_02_trajectories.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print(f"Random policy in-target rate: {((df['ph_t1']>=PH_LOW)&(df['ph_t1']<=PH_HIGH)).mean():.1%}")
print("Most trajectories drift but never converge — this is the gap the world model must close.")

## 5 · World Model — Learning the Reactor's Physics

The world model answers: *if the reactor is at pH $x$ and I add $d$ ml of base/acid,
what will the pH be at the next step?*

$$\hat{pH}_{t+1} = \hat{f}_\theta(pH_t, T_t, V_t, \text{buffer}, \text{dose\_ml})$$

We use a **Gradient Boosting Regressor** (200 trees) as the world model.
It captures the non-linear pH buffer dynamics without any physics assumptions.

**Why Gradient Boosting?**
- Handles non-linear interactions between pH, buffer, and dose
- Naturally captures the saturation effects near pH extremes
- Interpretable via feature importance

**Train/test split**: 80%/20% with `random_state=42`.


In [ ]:
# ── Train world model ────────────────────────────────────────────────────────
X = df[features_wm].values
y = df[target_wm].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

world_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
world_model.fit(X_train, y_train)

y_pred = world_model.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"World model trained on {X_train.shape[0]:,} transitions.")
print()
print(f"  R²  (test) = {r2:.4f}")
print(f"  MAE (test) = {mae:.4f} pH units")
print()

feat_imp = dict(zip(features_wm, world_model.feature_importances_.round(3)))
print("Feature importance:")
for feat, imp in sorted(feat_imp.items(), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<20} {imp:.3f}  {bar}")

In [ ]:
# ── World model validation ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor(C_BG)
fig.suptitle("World Model Validation — GBR (200 trees)", color="white", fontsize=13)

# Actual vs Predicted
ax = axes[0]
sample_idx = np.random.choice(len(y_test), 1000, replace=False)
ax.scatter(y_test[sample_idx], y_pred[sample_idx],
           alpha=0.3, s=8, color=C_BLUE, edgecolors="none")
ph_min, ph_max = y_test.min(), y_test.max()
ax.plot([ph_min, ph_max], [ph_min, ph_max], color=C_OK, lw=2, ls="--", label="Perfect prediction")
ax.set_xlabel("Actual pH(t+1)"); ax.set_ylabel("Predicted pH(t+1)")
ax.set_title(f"Actual vs Predicted\nR² = {r2:.4f} | MAE = {mae:.4f}", color="white")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Residuals distribution
ax = axes[1]
residuals = y_pred - y_test
ax.hist(residuals, bins=50, color=C_BLUE, edgecolor=C_BG, alpha=0.85)
ax.axvline(0, color=C_OK, lw=2, ls="--", label="Zero error")
ax.axvline(residuals.mean(), color=C_RED, lw=1.5, ls=":", label=f"Mean={residuals.mean():.4f}")
ax.set_xlabel("Prediction error (pH units)"); ax.set_ylabel("Count")
ax.set_title("Residual Distribution\nSymmetric → unbiased model", color="white")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Feature importance
ax = axes[2]
fi_sorted = sorted(zip(features_wm, world_model.feature_importances_), key=lambda x: -x[1])
names, imps = zip(*fi_sorted)
ax.barh(names, imps, color=C_BLUE, edgecolor=C_BG, alpha=0.85, height=0.55)
ax.set_xlabel("Feature importance")
ax.set_title("What Drives pH Next Step?\nWorld Model Feature Importance", color="white")
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("fig_03_world_model.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print(f"ph_t dominates (0.889) — the current pH is the strongest predictor of the next pH.")
print(f"dose_ml is second (0.107) — the dosing action has meaningful but secondary influence.")
print(f"Temperature and volume have negligible effect in this operating range.")

## 6 · The Reactor Environment

The real reactor is defined inline — no external import needed.
It uses the same physics as the data-generating process,
but with fresh random seeds so the planner cannot simply memorize trajectories.

The planner will use the **world model**, not this environment, to simulate futures.
It will then interact with this environment to collect real outcomes.


In [ ]:
# ── Chemical reactor environment (self-contained) ────────────────────────────
def dyn_ph(pH, temp, vol, dose, buf):
    # Real pH dynamics (stochastic, non-linear buffer).
    if dose == 0.0:
        return float(np.clip(pH + np.random.normal(0, 0.02), 0.0, 14.0))
    sign = np.sign(dose); intensity = abs(dose)
    dist = abs(pH - PH_TARGET) + 0.3
    eff  = intensity / (buf * dist)
    if sign > 0:
        sat = 1.0 / (1.0 + max(0.0, pH - PH_TARGET)); delta = eff * sat
    else:
        sat = 1.0 / (1.0 + max(0.0, PH_TARGET - pH)); delta = -eff * sat
    return float(np.clip(pH + delta + np.random.normal(0, 0.03), 0.0, 14.0))

def calc_reward(pH, dose):
    # Reward: penalize deviation, reward target zone, penalize reagent use.
    r = -10.0 * abs(pH - PH_TARGET)
    if PH_LOW <= pH <= PH_HIGH: r += 5.0
    r -= 0.2 * abs(dose)
    return float(r)

def fresh_state():
    # Sample a fresh episode starting state.
    return (float(np.clip(np.random.normal(5.5, 0.7), 3.0, 9.0)),
            np.random.uniform(20.0, 30.0),
            np.random.uniform(900.0, 1100.0),
            np.random.uniform(0.8, 1.5))

print("Reactor environment ready (inline, no external imports).")
print("  pH dynamics : non-linear with buffer resistance + Gaussian noise")
print("  Reward      : -10 × |pH-7| + 5 (if in target) - 0.2 × |dose|")

## 7 · Model-Based Planner — Lookahead Policy

The planner uses the world model to simulate $H$ steps ahead for each of the 5 possible actions,
computing the expected discounted return:

$$\hat{G}(a) = \sum_{h=0}^{H-1} \gamma^h \hat{r}_{t+h}(a)$$

The action with the highest $\hat{G}$ is executed in the real reactor.

**Key insight**: a horizon of $H=1$ outperforms longer horizons on this problem.

Why? The world model is very accurate for 1 step (MAE = 0.028 pH), but prediction errors
**compound** over multiple steps. At $H=5$, the simulated pH has drifted from reality
enough that the planner is optimizing a fictional future, not the real one.

This is a fundamental trade-off in Model-Based RL: **longer horizon ≠ better performance**.
The optimal horizon depends on world model accuracy and environment stochasticity.


In [ ]:
# ── Model-Based Planner ──────────────────────────────────────────────────────
def plan_action(pH, temp, vol, buf, model, horizon=1, gamma=0.95):
    # Select best action via H-step lookahead. Returns best_action_id (0-4).
    best_action, best_return = -1, -np.inf

    for action_id, dose in ACTIONS.items():
        total_return = 0.0
        ph_sim, vol_sim = pH, vol

        for h in range(horizon):
            # Simulate next pH via world model
            x_in = np.array([[ph_sim, temp, vol_sim, buf, dose]])
            ph_next = float(model.predict(x_in)[0])

            # Accumulate discounted reward
            total_return += (gamma ** h) * calc_reward(ph_next, dose)

            # Propagate state (deterministic world model)
            ph_sim  = ph_next
            vol_sim += dose * 0.5

        if total_return > best_return:
            best_return, best_action = total_return, action_id

    return best_action

# Quick test
pH0, temp0, vol0, buf0 = fresh_state()
a_star = plan_action(pH0, temp0, vol0, buf0, world_model, horizon=1)
print(f"Test state: pH={pH0:.2f}, T={temp0:.1f}°C, V={vol0:.0f}L, buf={buf0:.2f}")
print(f"Recommended action: [{a_star}] {ACTION_NAMES[a_star]}  (dose={ACTIONS[a_star]:+.1f} ml)")

## 8 · Evaluation — Model-Based vs. Random Policy

We run 100 fresh episodes with each policy and compare:
- **Mean reward per step**
- **Percentage of steps with pH in target [6.8, 7.2]**
- **pH trajectory quality** (visual comparison)


In [ ]:
# ── Evaluate both policies ───────────────────────────────────────────────────
np.random.seed(99)
N_EP, MAX_STEPS = 100, 25

def run_evaluation(policy_fn, n_ep=N_EP, max_steps=MAX_STEPS, label=""):
    step_rewards, in_target, ph_history = [], [], []
    np.random.seed(99)

    for _ in range(n_ep):
        pH, temp, vol, buf = fresh_state()
        ep_ph = [pH]

        for _ in range(max_steps):
            action = policy_fn(pH, temp, vol, buf)
            dose   = ACTIONS[action]
            pH_next = dyn_ph(pH, temp, vol, dose, buf)
            r = calc_reward(pH_next, dose)

            step_rewards.append(r)
            in_target.append(PH_LOW <= pH_next <= PH_HIGH)
            ep_ph.append(pH_next)
            pH = pH_next; vol += dose * 0.5
            if PH_LOW <= pH_next <= PH_HIGH:
                break

        ph_history.append(ep_ph)

    return {
        "mean_reward": np.mean(step_rewards),
        "in_target":   np.mean(in_target),
        "ph_history":  ph_history
    }

print("Running evaluations (100 episodes each)...")
res_rand = run_evaluation(
    lambda ph,t,v,b: np.random.randint(5),
    label="Random"
)
res_mb = run_evaluation(
    lambda ph,t,v,b: plan_action(ph,t,v,b,world_model,horizon=1),
    label="Model-Based H=1"
)

print()
print(f"{'Policy':<22} {'Mean Reward/Step':>18} {'Steps in Target':>16}")
print("=" * 58)
print(f"{'Random':<22} {res_rand['mean_reward']:>18.3f} {res_rand['in_target']:>15.1%}")
print(f"{'Model-Based H=1':<22} {res_mb['mean_reward']:>18.3f} {res_mb['in_target']:>15.1%}")
print()
rw_gain = res_mb['mean_reward'] - res_rand['mean_reward']
tgt_gain = (res_mb['in_target'] - res_rand['in_target']) * 100
print(f"Reward improvement   : {rw_gain:+.3f} per step")
print(f"Target rate gain     : {tgt_gain:+.1f} percentage points")
print(f"Target rate multiplier: {res_mb['in_target']/res_rand['in_target']:.1f}× vs random")

In [ ]:
# ── Trajectory comparison ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.patch.set_facecolor(C_BG)
fig.suptitle("pH Trajectories — Random Policy (top) vs Model-Based Planner (bottom)",
             color="white", fontsize=13, y=1.01)

for col, (res, label, color) in enumerate(
    [(res_rand, "Random", C_RED), (res_mb, "Model-Based H=1", C_OK)]
):
    for i in range(4):
        ax = axes[col, i]
        ep_ph = res["ph_history"][i]
        steps = range(len(ep_ph))
        ax.plot(steps, ep_ph, color=color, lw=2)
        ax.axhspan(PH_LOW, PH_HIGH, alpha=0.15, color=C_OK)
        ax.axhline(PH_TARGET, color=C_OK, lw=1, ls="--")
        ax.set_ylim(3, 10); ax.set_xlabel("Step", fontsize=8); ax.grid(True, alpha=0.3)
        in_tgt = sum(PH_LOW <= p <= PH_HIGH for p in ep_ph)
        ax.set_title(f"{label}\nEp {i+1} | In target: {in_tgt}/{len(ep_ph)} steps",
                     color="white", fontsize=8)
        ax.set_ylabel("pH", fontsize=8)

plt.tight_layout()
plt.savefig("fig_04_comparison.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Model-based trajectories converge toward pH=7 more reliably.")
print("Green shaded zone = [6.8, 7.2] target window.")

## 9 · Interpretability — What the Planner Decided

Two lenses on the model-based agent's decisions:

1. **Action frequency vs. pH** — does the planner correctly choose base when acidic,
   acid when alkaline, and no-dose near the target?
2. **Planning value landscape** — what expected return does the planner assign
   to each action across the pH spectrum?


In [ ]:
# ── Action selection across pH range ─────────────────────────────────────────
ph_test_range = np.linspace(3.5, 9.5, 60)
temp_ref, vol_ref, buf_ref = 25.0, 1000.0, 1.15

planner_actions = []
for ph in ph_test_range:
    a = plan_action(ph, temp_ref, vol_ref, buf_ref, world_model, horizon=1)
    planner_actions.append(a)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor(C_BG)

# Action map
action_colors = {0:"#8b949e", 1:C_OK, 2:"#00CC44", 3:C_WARN, 4:C_RED}
for ph, a in zip(ph_test_range, planner_actions):
    ax1.bar(ph, 1, width=0.1, color=action_colors[a], alpha=0.85)

ax1.axvline(PH_LOW, color="white", lw=1.5, ls="--", alpha=0.7)
ax1.axvline(PH_HIGH, color="white", lw=1.5, ls="--", alpha=0.7)
ax1.axvline(PH_TARGET, color=C_OK, lw=2)
ax1.set_xlabel("Current pH"); ax1.set_yticks([])
ax1.set_title("Planner Action Choice across pH\n(T=25°C, V=1000L, buf=1.15)",
              color="white", fontsize=10)

from matplotlib.patches import Patch
legend_e = [Patch(color=action_colors[i], label=n) for i,n in enumerate(ACTION_NAMES)]
ax1.legend(handles=legend_e, fontsize=7, loc="upper right")
ax1.grid(True, axis="x", alpha=0.3)

# Expected return per action across pH
for action_id, dose in ACTIONS.items():
    returns = []
    for ph in ph_test_range:
        x = np.array([[ph, temp_ref, vol_ref, buf_ref, dose]])
        ph_next = float(world_model.predict(x)[0])
        returns.append(calc_reward(ph_next, dose))
    ax2.plot(ph_test_range, returns, label=ACTION_NAMES[action_id],
             color=list(action_colors.values())[action_id], lw=1.8)

ax2.axvspan(PH_LOW, PH_HIGH, alpha=0.1, color=C_OK)
ax2.axvline(PH_TARGET, color=C_OK, lw=1.5, ls="--")
ax2.set_xlabel("Current pH"); ax2.set_ylabel("Expected reward (1 step)")
ax2.set_title("Expected Reward per Action\nModel decides: argmax over this", color="white", fontsize=10)
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_05_planner.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Observation: below pH~6.5, planner consistently adds large base (+1.0ml).")
print("Above pH~7.5, planner adds acid. Near target: no dose or small adjustments.")
print("This is chemically correct — the model learned the right control logic from data.")

## 10 · Statistical Validation

Two validation perspectives:

1. **Planning horizon sensitivity** — the compounding model error problem
2. **World model accuracy by pH range** — where is the model most/least reliable?


In [ ]:
# ── Planning horizon sensitivity ─────────────────────────────────────────────
print("Planning horizon sensitivity (100 episodes each):")
print(f"{'Horizon H':>10} {'Mean Reward':>14} {'In-Target':>12} {'Note':>20}")
print("-" * 60)

for h in [1, 2, 3, 5]:
    np.random.seed(99)
    res_h = run_evaluation(
        lambda ph,t,v,b,h=h: plan_action(ph,t,v,b,world_model,horizon=h)
    )
    note = "← optimal" if h==1 else ("model errors compound" if h>=3 else "")
    print(f"{h:>10} {res_h['mean_reward']:>14.3f} {res_h['in_target']:>11.1%} {note:>20}")

print()
print("Key insight: longer horizon ≠ better performance.")
print("At H=5, the world model's per-step errors compound across 5 steps,")
print("causing the planner to optimize a trajectory that diverges from reality.")

In [ ]:
# ── World model accuracy by pH range ─────────────────────────────────────────
df["ph_bin"]   = pd.cut(df["ph_t"], bins=[0,5,6,6.5,6.8,7.2,7.5,8,14],
                          labels=["<5","5-6","6-6.5","6.5-6.8","6.8-7.2","7.2-7.5","7.5-8",">8"])
df["pred_ph1"] = world_model.predict(df[features_wm].values)
df["abs_err"]  = (df["pred_ph1"] - df["ph_t1"]).abs()

mae_by_bin = df.groupby("ph_bin", observed=True)["abs_err"].agg(["mean","count"]).round(4)
mae_by_bin.columns = ["MAE (pH units)","Sample count"]
display(mae_by_bin)

fig, ax = plt.subplots(figsize=(9, 3.5))
fig.patch.set_facecolor(C_BG)
bins = mae_by_bin.index.astype(str)
maes = mae_by_bin["MAE (pH units)"].values
colors_b = [C_OK if b=="6.8-7.2" else C_BLUE for b in bins]
ax.bar(range(len(bins)), maes, color=colors_b, edgecolor=C_BG, alpha=0.85)
ax.set_xticks(range(len(bins))); ax.set_xticklabels(bins, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("MAE (pH units)"); ax.set_title("World Model MAE by pH Range", color="white", fontsize=11)
ax.axhline(mae_by_bin["MAE (pH units)"].mean(), color=C_RED, lw=1.5, ls="--",
           label=f"Overall MAE = {mae_by_bin['MAE (pH units)'].mean():.4f}")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig_06_model_accuracy.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("The world model is most accurate in the target zone [6.8-7.2] (green bar).")
print("This is where planning decisions matter most — and where model confidence is highest.")

## 11 · Simulator — Model-Based pH Control in Action

Three scenarios test the planner's behavior across different starting conditions:
strongly acidic, near-neutral, and strongly alkaline.


In [ ]:
# ── pH control simulator ─────────────────────────────────────────────────────
def control_ph(start_pH, temp=25.0, vol=1000.0, buf=1.15,
               max_steps=25, horizon=1, verbose=True):
    # Run model-based planner on real reactor. Returns control trajectory DataFrame.
    pH, history = start_pH, []

    for step in range(max_steps):
        action = plan_action(pH, temp, vol, buf, world_model, horizon=horizon)
        dose   = ACTIONS[action]
        pH_next = dyn_ph(pH, temp, vol, dose, buf)
        reward  = calc_reward(pH_next, dose)

        history.append({
            "step": step, "ph": pH, "action": action,
            "action_name": ACTION_NAMES[action], "dose_ml": dose,
            "ph_next": pH_next, "reward": round(reward, 2),
            "in_target": PH_LOW <= pH_next <= PH_HIGH
        })

        if verbose:
            status = "✅" if PH_LOW <= pH_next <= PH_HIGH else "  "
            print(f"  Step {step:2d} | pH={pH:.3f} → {pH_next:.3f} | "
                  f"Action: {ACTION_NAMES[action]:<16} | R={reward:+.2f} {status}")

        pH = pH_next; vol += dose * 0.5
        if PH_LOW <= pH_next <= PH_HIGH:
            print(f"  [TARGET REACHED at step {step}]")
            break

    return pd.DataFrame(history)

In [ ]:
# ── Scenario A: Strongly acidic start ────────────────────────────────────────
print("=" * 65)
print("SCENARIO A — Strongly acidic start (pH = 4.2)")
print("=" * 65)
traj_a = control_ph(start_pH=4.2, verbose=True)

# ── Scenario B: Near-neutral start ────────────────────────────────────────────
print()
print("=" * 65)
print("SCENARIO B — Near-neutral start (pH = 6.5)")
print("=" * 65)
traj_b = control_ph(start_pH=6.5, verbose=True)

# ── Scenario C: Alkaline start ────────────────────────────────────────────────
print()
print("=" * 65)
print("SCENARIO C — Alkaline start (pH = 8.5)")
print("=" * 65)
traj_c = control_ph(start_pH=8.5, verbose=True)

In [ ]:
# ── Visualize all three control trajectories ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(C_BG)
fig.suptitle("Model-Based pH Control — 3 Starting Conditions",
             color="white", fontsize=13, y=1.02)

scenario_data = [
    (traj_a, "Scenario A — Strongly Acidic (pH=4.2)", C_RED),
    (traj_b, "Scenario B — Near-Neutral (pH=6.5)",    C_WARN),
    (traj_c, "Scenario C — Alkaline (pH=8.5)",         C_BLUE),
]

for ax, (traj, title, color) in zip(axes, scenario_data):
    steps_plot = list(traj["step"]) + [traj["step"].iloc[-1]+1]
    ph_plot    = list(traj["ph"])   + [traj["ph_next"].iloc[-1]]
    ax.plot(steps_plot, ph_plot, color=color, lw=2.5, marker="o", markersize=5)
    ax.axhspan(PH_LOW, PH_HIGH, alpha=0.20, color=C_OK, label="Target zone")
    ax.axhline(PH_TARGET, color=C_OK, lw=1.5, ls="--", alpha=0.8)
    ax.set_xlabel("Step"); ax.set_ylabel("pH")
    in_target_steps = traj["in_target"].sum()
    ax.set_title(f"{title}\n{in_target_steps}/{len(traj)} steps in target",
                 color="white", fontsize=9)
    ax.set_ylim(3, 10); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_07_scenarios.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

In [ ]:
# ── GRAND FINAL REFLECTION — 24 Projects Complete ────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   PROJECT 24 — COMPLETE                                                     ║
║   THE PORTFOLIO — COMPLETE                                                  ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  Twenty-four projects.                                                       ║
║                                                                              ║
║  We began by asking: can machine learning serve operational excellence?      ║
║  The answer, across two dozen industrial problems, is not just yes —         ║
║  it is: yes, and in ways that classical statistics cannot reach.             ║
║                                                                              ║
║  We learned to SEE (supervised):                                            ║
║    Classification: defect detection, failure prediction                      ║
║    Regression: throughput modeling, cycle time estimation                    ║
║    Regularization: Lasso, Ridge, ElasticNet                                  ║
║                                                                              ║
║  We learned to DISCOVER (unsupervised):                                     ║
║    Clustering: process segmentation, quality grouping                        ║
║    Anomaly detection: Z-Score, Isolation Forest                              ║
║    Dimensionality reduction: pattern extraction at scale                     ║
║                                                                              ║
║  We learned to ACT (reinforcement):                                         ║
║    Q-Learning: the agent that memorized what worked                          ║
║    REINFORCE: the agent that followed the gradient of reward                 ║
║    Model-Based: the agent that learned to simulate before acting             ║
║                                                                              ║
║  The pH reactor in this final notebook closes the arc:                      ║
║  an agent that, given only a dataset of past transitions,                   ║
║  builds a model of how the world responds,                                  ║
║  simulates possible futures through that model,                             ║
║  and selects the action that leads closest to pH = 7.0.                    ║
║                                                                              ║
║  It does not wait to observe the consequences of a bad decision.            ║
║  It simulates them first.                                                   ║
║                                                                              ║
║  Three times more steps in target than random. Not from more data.          ║
║  Not from a more complex model. From planning.                              ║
║                                                                              ║
║  The portfolio is complete.                                                 ║
║  The journey continues.                                                     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")
print("LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa")